# 03 — RQ2: pose vs appearance for triage state classification
Full-scale version of `scripts/train_rq2_local.py`: train on multiple Okutama
videos, hold out whole videos (not just tracks), compare PoseMLP vs
AppearanceCNN, run the ablations.


### Setup (every notebook starts with this)
1. Runtime → Change runtime type → **T4 GPU** (free tier).
2. Zip your local `Project/` folder's `src/` and `scripts/` dirs as `src.zip`
   (`cd Project && zip -r src.zip src scripts`), then either upload it below
   or put it in Drive and adjust `SRC_ZIP`.


In [ ]:
# --- environment ---
!pip -q install ultralytics rtmlib onnxruntime-gpu
import torch, os
print('cuda:', torch.cuda.is_available())

# --- project code: upload src.zip (or mount Drive and set SRC_ZIP) ---
from pathlib import Path
SRC_ZIP = None  # e.g. '/content/drive/MyDrive/sar_project/src.zip'
if SRC_ZIP is None:
    from google.colab import files
    up = files.upload()  # choose src.zip
    SRC_ZIP = next(iter(up))
!mkdir -p /content/project && unzip -q -o "$SRC_ZIP" -d /content/project
import sys
sys.path.insert(0, '/content/project/src')
sys.path.insert(0, '/content/project')
print('project code ready')

# --- results go to Drive so they survive the session ---
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/sar_project_results'); OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Okutama-Action direct downloads (public Dropbox folder, verified working).
# preview=<file>&dl=1 selects a single file from the shared folder.
OKUTAMA_BASE = ('https://www.dropbox.com/scl/fo/9qvpsb3fsamvqzsa12149/'
                'APTyV-f01XLnJ0WFpZSBLOE?preview={name}&rlkey=7u7131amaul29amyr4jbnnu03&dl=1')

def fetch_okutama(name, dest='/content/data/okutama'):
    import subprocess, pathlib
    d = pathlib.Path(dest); d.mkdir(parents=True, exist_ok=True)
    zp = d / name
    if not zp.exists():
        subprocess.run(['curl', '-L', '-o', str(zp), OKUTAMA_BASE.format(name=name)], check=True)
    subprocess.run(['unzip', '-q', '-o', str(zp), '-d', str(d)], check=True)
    return d


In [ ]:
# Download videos (TrainSetVideos ~5GB — Colab disk is fine; start with Sample)
fetch_okutama('Sample.zip')
# fetch_okutama('TrainSetVideos.zip')   # uncomment for the full run
# fetch_okutama('TestSetVideos.zip')
import glob
videos = sorted(glob.glob('/content/data/okutama/**/*.mov', recursive=True))
labels = sorted(glob.glob('/content/data/okutama/**/*.txt', recursive=True))
print(len(videos), 'videos')


In [ ]:
import numpy as np, sys
import config
from pathlib import Path
config.MODELS_DIR = OUT; config.TABLES_DIR = OUT; config.FIGURES_DIR = OUT
from scripts.train_rq2_local import extract, build_dataset
from data.okutama import parse_annotations

all_tracks, all_crops, offset = {}, {}, 0
for v, t in zip(videos, labels):
    frames = parse_annotations(t)
    tr, cr = extract(v, frames, step=2, pose_device='cuda')
    # re-key track ids so videos don't collide; remember source video per track
    for tid, seq in tr.items():
        all_tracks[offset + tid] = seq
    for (tid, f), c in cr.items():
        all_crops[(offset + tid, f)] = c
    offset += 10_000
X, C, y, tids, classes = build_dataset(all_tracks, all_crops)


In [ ]:
# Split by VIDEO (tids//10_000) — stricter than by-track, matches the report
vid_of = tids // 10_000
val_vids = set(list(sorted(set(vid_of)))[-max(1, len(set(vid_of))//4):])
va = np.isin(vid_of, list(val_vids)); tr = ~va
print('train', tr.sum(), 'val', va.sum())

from classify import PoseMLP, AppearanceCNN, train_classifier, predict, evaluate
counts = np.bincount(y[tr], minlength=len(classes)).astype('float32')
w = counts.sum()/np.maximum(counts,1)/len(classes)

pose_model,_ = train_classifier(PoseMLP(len(classes)), X[tr], y[tr], X[va], y[va],
                                epochs=60, class_weights=w, device='cuda')
m_pose = evaluate(y[va], predict(pose_model, X[va], device='cuda'), classes, 'pose_mlp_full')

Cn = (C.astype('float32')/255.).transpose(0,3,1,2)
cnn,_ = train_classifier(AppearanceCNN(len(classes)), Cn[tr], y[tr], Cn[va], y[va],
                         epochs=60, class_weights=w, device='cuda')
m_app = evaluate(y[va], predict(cnn, Cn[va], device='cuda'), classes, 'appearance_cnn_full')

import pandas as pd
pd.DataFrame([{'model':'PoseMLP', **m_pose['per_class_f1'], 'macro_f1':m_pose['macro_f1']},
              {'model':'AppearanceCNN', **m_app['per_class_f1'], 'macro_f1':m_app['macro_f1']}]
             ).to_csv(OUT/'rq2_full.csv', index=False)


**Ablations to run** (rerun the two cells above, one change at a time):
window size 5/15/30; crop resolution 32/64/96; pose-features-only vs +temporal
(slice `X[:, :41]`); per-class error analysis — save 20 misclassified crops
and look at them. The last one is worth the most marks.
